# 🦴 Consistent Identity AI — Step 3
## Pose Control with ControlNet + DWPose

**Goal:** Extract skeleton keypoints from any reference image and use ControlNet
to control the body pose in generation — without changing who the person is.

**Where we are in the pipeline:**
```
[Ref Image] → [Identity Encoder ✅] → [Identity Tokens]
                                              ↓
[Pose Image] → [DWPose Extractor] → [Skeleton Map] → [ControlNet] → [Diffusion U-Net]
```

**What this notebook covers:**
- ✅ Step 3A: Install ControlNet + DWPose dependencies
- ✅ Step 3B: Extract body skeleton keypoints with DWPose
- ✅ Step 3C: Visualize the pose skeleton
- ✅ Step 3D: Load ControlNet (openpose) conditioned SDXL
- ✅ Step 3E: Generate an image from pose skeleton + text prompt
- ✅ Step 3F: Save pose map for use in Step 5 (full pipeline)

> ⚡ Requires GPU: Runtime → Change runtime type → T4 GPU

---
## 📦 Step 3A — Install Dependencies

In [ ]:
# ── Core (skip if already installed from Step 1+2) ────────────────────────────
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers transformers accelerate
!pip install -q opencv-python-headless Pillow matplotlib numpy

# ── ControlNet pipeline support ───────────────────────────────────────────────
!pip install -q controlnet-aux   # DWPose, OpenPose, Canny, etc.
!pip install -q einops timm      # required by DWPose internals

# ── For downloading model weights ─────────────────────────────────────────────
!pip install -q huggingface_hub

print('✅ Dependencies installed.')

---
## 🔧 Step 3B — Imports & GPU Check

In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path

from controlnet_aux import DWposeDetector
from diffusers import (
    StableDiffusionXLControlNetPipeline,
    ControlNetModel,
    AutoencoderKL,
)
from diffusers.utils import load_image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## 🦴 Step 3C — DWPose Skeleton Extractor

**DWPose** detects 18 body keypoints + 21 hand keypoints + 68 face keypoints.
It produces a clean stick-figure skeleton image that ControlNet uses as its
conditioning signal — telling the model *how the body is positioned* without
carrying any identity information.

In [ ]:
class PoseExtractor:
    """
    Extracts body pose skeleton from an image using DWPose.

    DWPose detects:
        - 18 body keypoints  (shoulders, elbows, wrists, hips, knees, ankles...)
        - 21 hand keypoints  (each hand)
        - 68 face landmarks  (optional)

    Output is a rendered skeleton image (black background + coloured joints/limbs)
    that ControlNet uses as spatial conditioning.
    """

    # COCO 18-keypoint skeleton connections
    # Each tuple: (keypoint_a, keypoint_b, RGB_color)
    SKELETON_CONNECTIONS = [
        (0,  1,  (255, 0,   0  )),  # nose  → neck
        (1,  2,  (255, 85,  0  )),  # neck  → right shoulder
        (2,  3,  (255, 170, 0  )),  # right shoulder → right elbow
        (3,  4,  (255, 255, 0  )),  # right elbow → right wrist
        (1,  5,  (170, 255, 0  )),  # neck  → left shoulder
        (5,  6,  (85,  255, 0  )),  # left shoulder → left elbow
        (6,  7,  (0,   255, 0  )),  # left elbow → left wrist
        (1,  8,  (0,   255, 85 )),  # neck  → right hip
        (8,  9,  (0,   255, 170)),  # right hip → right knee
        (9,  10, (0,   255, 255)),  # right knee → right ankle
        (1,  11, (0,   170, 255)),  # neck  → left hip
        (11, 12, (0,   85,  255)),  # left hip → left knee
        (12, 13, (0,   0,   255)),  # left knee → left ankle
        (0,  14, (85,  0,   255)),  # nose  → right eye
        (14, 16, (170, 0,   255)),  # right eye → right ear
        (0,  15, (255, 0,   255)),  # nose  → left eye
        (15, 17, (255, 0,   170)),  # left eye → left ear
    ]

    def __init__(self, device='cuda'):
        self.device = device
        # DWposeDetector auto-downloads weights on first use (~200MB)
        self.detector = DWposeDetector()
        if hasattr(self.detector, 'to'):
            self.detector = self.detector.to(device)
        print('✅ DWPose detector loaded')

    def extract(self, image: Image.Image, output_size: tuple = None) -> dict:
        """
        Extract pose from image.

        Args:
            image      : PIL Image (RGB)
            output_size: (width, height) to resize output, or None to keep original

        Returns dict with:
            'pose_image'  : PIL Image — rendered skeleton on black background
            'keypoints'   : list of (x, y) tuples, None if keypoint not detected
            'detected'    : bool — whether a person was found
        """
        if output_size:
            image = image.resize(output_size, Image.LANCZOS)

        # DWposeDetector returns the rendered skeleton image directly
        pose_image = self.detector(image, include_body=True, include_hand=True, include_face=False)

        # Check if any pose was detected (non-black pixels)
        pose_arr = np.array(pose_image)
        detected = pose_arr.sum() > 1000  # threshold: at least some skeleton pixels

        if not detected:
            print('⚠️  No person/pose detected in image.')

        return {
            'pose_image': pose_image,
            'detected'  : detected,
            'size'      : pose_image.size,
        }

    def extract_from_path(self, image_path: str, output_size: tuple = None) -> dict:
        """Convenience wrapper — load from file path."""
        image = Image.open(image_path).convert('RGB')
        return self.extract(image, output_size=output_size)

    def save_pose(self, pose_result: dict, path: str):
        """Save the pose skeleton image as PNG."""
        pose_result['pose_image'].save(path)
        print(f'💾 Pose skeleton saved to {path}')


# ── Load the pose extractor ────────────────────────────────────────────────────
pose_extractor = PoseExtractor(device=device)

---
## 🖼️ Step 3D — Test Pose Extraction

Upload a photo of a person (full body works best) to see the skeleton.

In [ ]:
# ── Option A: Upload your own image ───────────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# image_path = list(uploaded.keys())[0]
# input_image = Image.open(image_path).convert('RGB')

# ── Option B: Load from URL (replace with a real full-body person photo) ───────
# input_image = load_image('https://your-image-url-here.jpg').convert('RGB')

# ── Option C: Synthetic placeholder ───────────────────────────────────────────
# Creates a basic stick figure manually for testing without a real photo
def make_test_person_image(size=512):
    """Draw a simple stick figure as a placeholder test image."""
    img = Image.new('RGB', (size, size), (240, 230, 210))
    d = ImageDraw.Draw(img)
    cx = size // 2
    # Head
    d.ellipse([cx-30, 40, cx+30, 100], outline=(80,60,40), width=3)
    # Body
    d.line([cx, 100, cx, 260], fill=(80,60,40), width=4)
    # Arms
    d.line([cx, 140, cx-80, 200], fill=(80,60,40), width=4)
    d.line([cx, 140, cx+80, 200], fill=(80,60,40), width=4)
    # Legs
    d.line([cx, 260, cx-60, 380], fill=(80,60,40), width=4)
    d.line([cx, 260, cx+60, 380], fill=(80,60,40), width=4)
    return img

input_image = make_test_person_image(512)
print('ℹ️  Using synthetic placeholder. Uncomment Option A or B to use a real photo.')
print(f'   Image size: {input_image.size}')

In [ ]:
# ── Extract pose ───────────────────────────────────────────────────────────────
pose_result = pose_extractor.extract(input_image, output_size=(768, 1024))

# ── Visualize side by side ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 7))
fig.suptitle('Pose Extraction — DWPose', fontsize=14, fontweight='bold')

axes[0].imshow(input_image)
axes[0].set_title('Input image')
axes[0].axis('off')

axes[1].imshow(pose_result['pose_image'])
axes[1].set_title(f'Extracted skeleton  |  detected: {pose_result["detected"]}')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('pose_extraction.png', dpi=120, bbox_inches='tight')
plt.show()

# Save pose map for Step 5
pose_extractor.save_pose(pose_result, 'my_pose.png')

---
## 🤖 Step 3E — Load ControlNet + SDXL

We load:
- **ControlNet** (openpose-sdxl) — the pose-conditioning adapter
- **SDXL base** — the diffusion backbone
- **VAE** — improved decoder for sharper results

> ⚠️ This downloads ~6–8 GB of model weights. It takes 5–10 min on first run.
> Colab caches to `/root/.cache/huggingface` between sessions.

In [ ]:
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, AutoencoderKL
import torch

print('Loading ControlNet (openpose-sdxl)...')
controlnet = ControlNetModel.from_pretrained(
    'thibaud/controlnet-openpose-sdxl-1.0',
    torch_dtype=torch.float16,
)

print('Loading VAE...')
vae = AutoencoderKL.from_pretrained(
    'madebyollin/sdxl-vae-fp16-fix',   # FP16-stable VAE for better quality
    torch_dtype=torch.float16,
)

print('Loading SDXL base pipeline...')
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
)

# Memory optimizations for T4 (16GB VRAM)
pipe.enable_model_cpu_offload()   # moves layers to CPU when not in use
try:
    pipe.enable_xformers_memory_efficient_attention()  # saves ~30% VRAM
except Exception as exc:
    print(f'Warning: xformers optimization unavailable: {exc}')

print('✅ Pipeline loaded and ready!')

---
## 🎨 Step 3F — Generate from Pose + Text Prompt

This is the key test: the pose skeleton controls *where* the body is,
while the text prompt controls *what* the person is wearing and the scene.

In Step 5 we'll also inject the **identity tokens** so the *face* stays consistent.

In [ ]:
class PoseConditionedGenerator:
    """
    Generates images conditioned on a pose skeleton and a text prompt.

    In Step 5, this will be extended with identity token injection
    so the generated person always looks like the reference subject.
    """

    def __init__(self, pipeline):
        self.pipe = pipeline

    def generate(
        self,
        pose_image      : Image.Image,
        prompt          : str,
        negative_prompt : str = None,
        num_steps       : int = 30,
        guidance_scale  : float = 7.5,
        controlnet_scale: float = 1.0,
        seed            : int = 42,
        width           : int = 768,
        height          : int = 1024,
    ) -> Image.Image:
        """
        Args:
            pose_image       : skeleton image from PoseExtractor
            prompt           : describe outfit, scene, lighting
            negative_prompt  : what to avoid in the output
            num_steps        : denoising steps (20–40 is typical)
            guidance_scale   : prompt adherence (7–10 for strong guidance)
            controlnet_scale : how strongly the pose is enforced
                               1.0 = full adherence, 0.5 = loose adherence
            seed             : for reproducibility
            width / height   : output resolution (must match pose_image size)
        """
        if negative_prompt is None:
            negative_prompt = (
                'blurry, deformed, disfigured, bad anatomy, extra limbs, '
                'missing limbs, floating limbs, mutated hands, poorly drawn hands, '
                'bad proportions, ugly, lowres, text, watermark'
            )

        generator = torch.Generator(device='cpu').manual_seed(seed)

        # Ensure pose image matches output resolution
        pose_resized = pose_image.resize((width, height), Image.LANCZOS)

        output = self.pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image=pose_resized,              # ControlNet conditioning
            num_inference_steps=num_steps,
            guidance_scale=guidance_scale,
            controlnet_conditioning_scale=controlnet_scale,
            generator=generator,
            width=width,
            height=height,
        )

        return output.images[0]


generator = PoseConditionedGenerator(pipe)
print('✅ PoseConditionedGenerator ready')

In [ ]:
# ── Run generation ─────────────────────────────────────────────────────────────
# Try different prompts — the POSE stays the same, only outfit/scene changes

prompts = [
    'a young woman in a red summer dress, standing in a park, golden hour lighting, photorealistic',
    'a young woman in a blue denim jacket and white jeans, urban street background, photorealistic',
    'a young woman in a formal black blazer, office background, professional headshot style',
]

results = []
for i, prompt in enumerate(prompts):
    print(f'Generating {i+1}/{len(prompts)}: {prompt[:60]}...')
    img = generator.generate(
        pose_image=pose_result['pose_image'],
        prompt=prompt,
        controlnet_scale=0.9,  # slightly loose for natural look
        seed=42 + i,
    )
    results.append((prompt, img))
    print(f'  ✓ Done')

print('\n✅ All generations complete!')

In [ ]:
# ── Visualize all results ──────────────────────────────────────────────────────
n = len(results) + 1
fig, axes = plt.subplots(1, n, figsize=(5 * n, 7))
fig.suptitle('Same Pose → Different Outfits', fontsize=14, fontweight='bold')

# First column: the pose skeleton
axes[0].imshow(pose_result['pose_image'])
axes[0].set_title('Pose skeleton\n(ControlNet input)', fontsize=10)
axes[0].axis('off')

# Remaining columns: generated images
for i, (prompt, img) in enumerate(results):
    axes[i+1].imshow(img)
    # Shorten prompt for display
    short = prompt[:50] + '...' if len(prompt) > 50 else prompt
    axes[i+1].set_title(short, fontsize=8, wrap=True)
    axes[i+1].axis('off')
    # Save individual results
    img.save(f'pose_gen_{i+1}.png')

plt.tight_layout()
plt.savefig('pose_control_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Results saved.')

---
## 🎛️ Step 3G — ControlNet Scale Experiment

This shows how `controlnet_conditioning_scale` affects the trade-off between
**pose fidelity** and **image naturalness**.

In [ ]:
scales = [0.4, 0.7, 1.0]
scale_results = []

base_prompt = 'a woman in a stylish white blouse, photorealistic, sharp focus, studio lighting'

for scale in scales:
    print(f'  ControlNet scale = {scale}...')
    img = generator.generate(
        pose_image=pose_result['pose_image'],
        prompt=base_prompt,
        controlnet_scale=scale,
        seed=99,
    )
    scale_results.append((scale, img))

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 7))
fig.suptitle('ControlNet Scale Comparison', fontsize=13, fontweight='bold')
descriptions = ['Loose pose (0.4)', 'Balanced (0.7)', 'Strict pose (1.0)']

for i, ((scale, img), desc) in enumerate(zip(scale_results, descriptions)):
    axes[i].imshow(img)
    axes[i].set_title(f'{desc}\nscale = {scale}', fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('controlnet_scale_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nTip: scale 0.7–0.9 is usually the sweet spot for natural-looking results.')

---
## 💾 Step 3H — Save Everything for Step 4

Save the pose image and pipeline state so we can load them in the next step
without re-downloading models.

In [ ]:
import json

# Save pose skeleton
pose_result['pose_image'].save('my_pose.png')

# Save generation config for reproducibility
config = {
    'controlnet_model' : 'thibaud/controlnet-openpose-sdxl-1.0',
    'sdxl_model'       : 'stabilityai/stable-diffusion-xl-base-1.0',
    'vae_model'        : 'madebyollin/sdxl-vae-fp16-fix',
    'pose_image'       : 'my_pose.png',
    'identity_file'    : 'my_identity.npz',  # from Step 2
    'recommended_scale': 0.85,
    'output_size'      : [768, 1024],
}
with open('pipeline_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('✅ Step 3 complete!')
print('   Saved: my_pose.png')
print('   Saved: pipeline_config.json')
print()
print('   Next → Step 4: Garment encoder (outfit reference image → texture features)')

---
## 📋 Summary

| Component | What it does |
|-----------|-------------|
| `PoseExtractor` | DWPose → 18-point skeleton from any photo |
| `ControlNetModel` | Injects pose skeleton into SDXL U-Net |
| `PoseConditionedGenerator` | Generates images from skeleton + text |
| `controlnet_conditioning_scale` | Controls pose strictness (0.7–0.9 recommended) |

## Key insight so far

```
Pose skeleton  →  controls WHERE the body is
Text prompt    →  controls WHAT they wear + scene
Identity tokens (Step 2) →  controls WHO they are  ← added in Step 5
Garment image  (Step 4)  →  controls exact outfit texture
```

Right now the generated person is **random** — same pose, same outfit,
but a different face every run. Step 5 locks the face to the reference subject.